# Tutorial 1: Introduction to AnnData and Exploratory Spatial Analysis

**Systems Biology - Spatial Proteomics Module**  
**University of Warsaw, Faculty of Mathematics, Informatics and Mechanics**

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Understand the structure and purpose of AnnData objects for spatial single-cell data
2. Load and explore Imaging Mass Cytometry (IMC) datasets
3. Analyze cell type distributions across tissue images
4. Transform and visualize protein marker expressions
5. Perform dimensionality reduction using PCA to identify image-level patterns
6. Interpret biological insights from computational analyses

---

## Biological Context

### Why Does Spatial Organization Matter?

In cancer biology, understanding *where* cells are located is as important as knowing *what* types of cells are present. The tumor microenvironment is a complex ecosystem where:

- **Tumor cells** interact with immune cells, determining treatment response
- **Immune infiltration patterns** predict patient survival
- **Spatial niches** create local microenvironments with distinct properties
- **Cell-cell communication** requires physical proximity

### Imaging Mass Cytometry (IMC)

IMC is a spatial proteomics technology that simultaneously measures:
- **40+ protein markers** per cell
- **Spatial coordinates** (X, Y position) in tissue sections
- **Morphological features** (cell size, shape)

Unlike traditional single-cell RNA sequencing (scRNA-seq), IMC preserves the tissue architecture, allowing us to study spatial relationships.

---

## Dataset Overview

In this tutorial, we work with an IMC dataset from cancer patients containing:
- **253,433 cells** measured across **132 tissue images**
- **40 protein markers** including immune markers (CD4, CD8, PD1) and tumor markers (Ecad)
- **14 cell types** identified: Tumor, CD4 T cells, CD8 T cells, Macrophages, B cells, etc.

Our goal: understand how cells are organized and what patterns exist across different patients.

## Part 1: Environment Setup and Data Loading

First, we import the necessary Python libraries for our analysis.

In [ ]:
# Data handling
import anndata
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.decomposition import PCA

# Display settings
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("All libraries imported successfully!")

### Loading the Dataset

Now we load the IMC dataset stored in HDF5-based AnnData format (.h5ad file).

In [ ]:
# Load the training dataset
DATA_PATH = 'data/train_adata.h5ad'
adata = anndata.read_h5ad(DATA_PATH)

# Display basic information
print(adata)

## Part 2: Understanding Ann Data Structure

AnnData (Annotated Data) is the standard format for storing single-cell and spatial omics data. Understanding its structure is crucial for effective analysis.

### Core Components of AnnData

An AnnData object has several key attributes:

1. **`.X`** - Main data matrix (cells × features)
   - Rows are observations (cells)
   - Columns are variables (genes/proteins)
   
2. **`.obs`** - DataFrame with cell metadata
   - Each row corresponds to a cell
   - Contains annotations like cell type, spatial position, quality metrics
   
3. **`.var`** - DataFrame with feature metadata
   - Each row corresponds to a measured feature (protein marker)
   - Contains marker names, channels, etc.
   
4. **`.layers`** - Dictionary of alternative data matrices
   - Same shape as `.X` but different transformations
   - Example: raw counts, normalized, transformed
   
5. **`.obsm`** - Multi-dimensional cell annotations
   - Stores matrices associated with cells (e.g., spatial coordinates, PCA, UMAP)
   
6. **`.obsp`** - Pairwise cell relationships
   - Stores cell-cell graphs (e.g., spatial neighborhood networks)
   
7. **`.uns`** - Unstructured annotations
   - Stores any additional metadata (parameters, results, etc.)

Let's explore each of these components in our dataset.

In [ ]:
# Check available layers
print("Available data layers:", list(adata.layers.keys()))
print("\nExpression matrix shape:", adata.layers['exprs'].shape)
print("This means:", adata.n_obs, "cells ×", adata.n_vars, "markers")

In [ ]:
# Task 1.4: Cell type distribution
celltype_counts = adata.obs['celltypes'].value_counts()

plt.figure(figsize=(12, 6))
celltype_counts.plot(kind='bar', color='steelblue')
plt.title('Cell Type Distribution Across All Images', fontsize=14, fontweight='bold')
plt.xlabel('Cell Type', fontsize=12)
plt.ylabel('Number of Cells', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nCell counts by type:")
print(celltype_counts)

In [ ]:
# Get Ecad expression
marker_name = 'Ecad'
marker_idx = np.where(adata.var['marker'] == marker_name)[0][0]

# Get raw and transformed values
raw_expr = adata.layers['exprs'][:, marker_idx]
transformed_expr = adata.layers['exprs_arcsinh'][:, marker_idx]

# Get cell types for coloring
is_tumor = adata.obs['celltypes'] == 'Tumor'

# Create comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw expression
axes[0].hist(raw_expr[~is_tumor], bins=100, alpha=0.5, label='Non-Tumor', color='blue')
axes[0].hist(raw_expr[is_tumor], bins=100, alpha=0.5, label='Tumor', color='red')
axes[0].set_title(f'{marker_name} - Raw Expression', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Raw Intensity', fontsize=10)
axes[0].set_ylabel('Number of Cells (log scale)', fontsize=10)
axes[0].set_yscale('log')
axes[0].legend()

# Transformed expression
axes[1].hist(transformed_expr[~is_tumor], bins=100, alpha=0.5, label='Non-Tumor', color='blue')
axes[1].hist(transformed_expr[is_tumor], bins=100, alpha=0.5, label='Tumor', color='red')
axes[1].set_title(f'{marker_name} - arcsinh Transformed', fontsize=12, fontweight='bold')
axes[1].set_xlabel('arcsinh(Intensity / 5)', fontsize=10)
axes[1].set_ylabel('Number of Cells (log scale)', fontsize=10)
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Observation: Ecad expression is higher in Tumor cells (red) than in other cell types (blue)")
print(f"The transformation makes the distribution more symmetric and easier to analyze.")

In [ ]:
# Task 2.1: Average proportions
avg_proportions = image_celltype_proportions.mean(axis=0).sort_values(ascending=False)
print("Average cell type proportions across all images:")
print(avg_proportions)
print(f"\nMost abundant cell type: {avg_proportions.idxmax()} ({avg_proportions.max():.2%})")

# Task 2.2: Image with highest CD8 proportion
cd8_max_image = image_celltype_proportions['CD8'].idxmax()
cd8_max_prop = image_celltype_proportions['CD8'].max()
print(f"\nImage with highest CD8 proportion: {cd8_max_prop:.2%}")

# Task 2.3: Tumor vs CD8 scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(
    image_celltype_proportions['Tumor'],
    image_celltype_proportions['CD8'],
    alpha=0.6,
    s=50,
    color='steelblue'
)
plt.xlabel('Tumor Cell Proportion', fontsize=12)
plt.ylabel('CD8 T Cell Proportion', fontsize=12)
plt.title('Relationship Between Tumor and CD8 T Cell Proportions', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = image_celltype_proportions[['Tumor', 'CD8']].corr().iloc[0, 1]
print(f"\nCorrelation between Tumor and CD8 proportions: {correlation:.3f}")

In [ ]:
# Task 3.1: Highest loading on PC1
pc1_max_celltype = loadings_df['PC1'].idxmax()
pc1_max_value = loadings_df['PC1'].max()
print(f"Task 3.1:")
print(f"Cell type with highest PC1 loading: {pc1_max_celltype} ({pc1_max_value:.3f})")
print(f"This suggests PC1 separates images by {pc1_max_celltype} abundance\n")

# Task 3.2: Components for 90% variance
n_for_90 = np.argmax(cumulative_var >= 0.9) + 1
print(f"Task 3.2:")
print(f"Number of components needed for ≥90% variance: {n_for_90}")
print(f"Actual cumulative variance: {cumulative_var[n_for_90-1]:.2%}\n")

# Task 3.3: PC2 vs PC3 scatter
plt.figure(figsize=(10, 8))
plt.scatter(pca_embedding[:, 1], pca_embedding[:, 2], alpha=0.6, s=80, color='coral', edgecolors='black', linewidth=0.5)
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]:.1%} variance)', fontsize=12)
plt.title('Images Projected onto PC2 and PC3', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.axhline(y=0, color='k', linewidth=0.5, alpha=0.5)
plt.axvline(x=0, color='k', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

print("Task 3.3:")
print("PC2 vs PC3 reveals different axes of variation in the data")

## References and Further Reading

1. **AnnData Documentation**: https://anndata.readthedocs.io/
2. **Imaging Mass Cytometry**: Giesen et al. (2014). "Highly multiplexed imaging of tumor tissues with subcellular resolution by mass cytometry." Nature Methods.
3. **Squidpy**: Palla et al. (2022). "Squidpy: a scalable framework for spatial omics analysis." Nature Methods.
4. **Hellinger Distance**: A metric for comparing probability distributions, particularly suited for compositional data.
5. **PCA**: Jolliffe, I. T. (2002). Principal Component Analysis. Springer Series in Statistics.

---

**End of Tutorial 1**

Save your work and make sure you understand the concepts before proceeding to Tutorial 2.

## Summary and Key Takeaways

### What We Learned

1. **AnnData Structure**
   - `.obs` stores cell-level metadata (position, cell type)
   - `.var` stores feature metadata (markers)
   - `.layers` stores different transformations of expression data
   - `.obsm` will store embeddings and spatial coordinates
   - `.obsp` will store cell-cell relationships (next tutorial)

2. **Data Transformations**
   - arcsinh transformation handles heavy-tailed distributions in IMC data
   - Hellinger transform (square root) is appropriate for compositional data

3. **Image-Level Analysis**
   - Cell type histograms summarize tissue composition
   - Normalization enables comparison across images
   - Hierarchical clustering reveals groups of similar images

4. **Dimensionality Reduction with PCA**
   - Reduces high-dimensional data to interpretable axes
   - Loadings show which features (cell types) drive each component
   - Explained variance quantifies information retention
   - First few PCs capture major biological variation

### Biological Insights

From this exploratory analysis, we observed:
- Images vary substantially in cell type composition
- Some images are tumor-dominated, others are immune-rich
- PCA reveals natural groupings based on microenvironment composition
- Cell type proportions contain meaningful biological signal

### Next Steps

In the next tutorial, we will:
- Explore **spatial coordinates** and visualize tissue architecture
- Build **spatial neighborhood graphs** using Squidpy
- Analyze **cell-cell interactions** and spatial patterns
- Identify **spatial niches** where specific cell types co-localize

---

## Additional Challenges (Optional)

For students who finish early or want extra practice:

**Challenge 1:** Repeat the PCA analysis using raw proportions (without Hellinger transform). Compare the results. Which cell types dominate the first PC?

**Challenge 2:** Use K-Means clustering to group images into 3-5 clusters based on their cell type composition. Characterize each cluster biologically.

**Challenge 3:** Investigate whether images from the same patient cluster together in PCA space by coloring points by `sample_id`.

**Challenge 4:** Create a violin plot comparing marker expression distributions (e.g., PD1, Ki67) across different cell types.

### Exercise 3: PCA Interpretation

**Task 3.1:** Which cell type has the highest (most positive) loading on PC1? What does this tell us about PC1?

**Task 3.2:** How many components would we need to explain at least 90% of the variance?

**Task 3.3:** Create a scatter plot of PC2 vs PC3. Does the structure change compared to PC1 vs PC2?

In [ ]:
# Create PCA scatter plot
plt.figure(figsize=(10, 8))
plt.scatter(pca_embedding[:, 0], pca_embedding[:, 1], alpha=0.6, s=80, color='steelblue', edgecolors='black', linewidth=0.5)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Images Projected onto First Two Principal Components', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.axhline(y=0, color='k', linewidth=0.5, alpha=0.5)
plt.axvline(x=0, color='k', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Each point represents one tissue image")
print("- Distance between points reflects similarity in cell type composition")
print("- Clusters suggest groups of images with similar microenvironments")

### Visualizing Images in PC Space

Let's plot images in the reduced 2D space defined by PC1 and PC2.

In [ ]:
# Create a DataFrame of PC loadings
loadings_df = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(n_components)],
    index=image_celltype_hellinger.columns
)

# Visualize loadings as heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    loadings_df,
    cmap='RdBu_r',
    center=0,
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'Loading'},
    linewidths=0.5
)
plt.title('PCA Loadings: Cell Type Contributions to Each PC', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component', fontsize=12)
plt.ylabel('Cell Type', fontsize=12)
plt.tight_layout()
plt.show()

print("\nBiological Interpretation:")
print("- Positive loadings: cell types that increase along this PC")
print("- Negative loadings: cell types that decrease along this PC")
print("- Large absolute values: cell types that strongly define this axis")

### Interpreting Principal Components

Each PC is a linear combination of cell types. The **loadings** tell us which cell types contribute most to each component.

In [ ]:
# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance per component
axes[0].bar(range(1, n_components + 1), pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Variance Explained by Each PC', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(1, n_components + 1))
axes[0].grid(axis='y', alpha=0.3)

# Cumulative variance
cumulative_var = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, n_components + 1), cumulative_var, marker='o', linewidth=2, markersize=8, color='steelblue')
axes[1].axhline(y=0.9, color='red', linestyle='--', label='90% threshold', alpha=0.7)
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Variance Explained', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(1, n_components + 1))
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("Explained variance ratio per component:")
for i, var in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var:.2%}")
print(f"\nCumulative variance explained by first {n_components} PCs: {cumulative_var[-1]:.2%}")

### Explained Variance

How much information does each principal component capture?

In [ ]:
# Fit PCA with 5 components
n_components = 5
pca = PCA(n_components=n_components)
pca_embedding = pca.fit_transform(image_celltype_hellinger)

print(f"PCA fitted with {n_components} components")
print(f"PCA embedding shape: {pca_embedding.shape}")
print(f"({pca_embedding.shape[0]} images × {pca_embedding.shape[1]} components)")

### Applying PCA

Now we fit PCA to find the principal components (directions of maximum variance) in the Hellinger-transformed data.

In [ ]:
# Apply Hellinger transform
image_celltype_hellinger = np.sqrt(image_celltype_proportions)

print("Hellinger-transformed cell type data:")
print(f"Shape: {image_celltype_hellinger.shape}")
print("\nFirst 5 images:")
print(image_celltype_hellinger.head())

## Part 5: Principal Component Analysis (PCA)

With 14 cell types, our image-level data is 14-dimensional. Principal Component Analysis (PCA) helps us:

1. **Reduce dimensionality** - Find the most important axes of variation
2. **Visualize patterns** - Plot images in 2D while preserving relationships
3. **Identify biological drivers** - Understand which cell types drive image similarity

### The Hellinger Transform

Before PCA, we apply the **Hellinger transform** (square root of proportions). Why?

- **Statistical motivation**: The Hellinger distance is appropriate for compositional (proportion) data
- **Practical effect**: Reduces the impact of dominant cell types, gives more weight to rare types
- **Formula**: Transform each proportion p to √p

### Exercise 2: Image-Level Analysis

**Task 2.1:** Calculate the average cell type proportion across all images. Which cell type is most abundant on average?

**Task 2.2:** Find the image with the highest proportion of CD8 T cells. What percentage of that image is CD8+ cells?

**Task 2.3:** Create a scatter plot showing Tumor proportion (x-axis) vs CD8 proportion (y-axis) for all images. Is there a relationship?

In [ ]:
# Create a clustered heatmap
plt.figure(figsize=(14, 10))
sns.clustermap(
    image_celltype_proportions,
    cmap='viridis',
    figsize=(12, 10),
    yticklabels=False,  # Too many images to show labels
    xticklabels=True,
    cbar_kws={'label': 'Cell Type Proportion'}
)
plt.suptitle('Cell Type Proportions Across All Images (Hierarchically Clustered)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Interpretation:")
print("- Similar images cluster together (similar cell type compositions)")
print("- Tumor-dominated images cluster separately from immune-rich images")
print("- Hierarchical clustering reveals natural groupings")

### Visualizing Cell Type Composition Across Images

Let's create a heatmap to visualize cell type proportions across all images.

In [ ]:
# Normalize each row to sum to 1
image_celltype_proportions = image_celltype_counts.div(image_celltype_counts.sum(axis=1), axis=0)

print("Normalized cell type proportions:")
print("\nFirst 5 images:")
print(image_celltype_proportions.head())

# Verify normalization
print(f"\nRow sums (should all be 1.0):")
print(image_celltype_proportions.sum(axis=1).head())

### Normalizing Cell Type Counts

Images may have different numbers of cells. To enable fair comparison, we normalize counts so that each row (image) sums to 1. This gives us the **cell type proportions** per image.

In [ ]:
# Create a pivot table: images × cell types
image_celltype_counts = pd.pivot_table(
    adata.obs,
    index='image',
    columns='celltypes',
    aggfunc='size',
    fill_value=0
)

print(f"Shape: {image_celltype_counts.shape}")
print(f"({image_celltype_counts.shape[0]} images × {image_celltype_counts.shape[1]} cell types)")
print("\nFirst 5 images:")
image_celltype_counts.head()

### Creating Cell Type Histogram per Image

We'll create a matrix where:
- **Rows** = images
- **Columns** = cell types
- **Values** = count of each cell type in each image

## Part 4: Image-Level Cell Type Analysis

So far, we've analyzed individual cells. Now let's aggregate data at the **image level** to understand tissue-level patterns. This is a common first step in spatial analysis: representing each image by its cell type composition.

### Comparing Raw vs Transformed Expressions

Let's visualize how the transformation affects Ecad (tumor marker) expression distribution.

In [ ]:
# Apply arcsinh transformation
adata.layers['exprs_arcsinh'] = np.arcsinh(adata.layers['exprs'] / 5)

print("Added new layer 'exprs_arcsinh' to AnnData object")
print(f"Available layers: {list(adata.layers.keys())}")

### arcsinh Transformation

The inverse hyperbolic sine (arcsinh) transformation is commonly used for mass cytometry data because it:

1. **Compresses large values** while preserving zero
2. **Stabilizes variance** across the dynamic range
3. **Approximates log transformation** for large values but handles zeros gracefully

The standard formula is: `arcsinh(x / 5)`

Let's apply this transformation and add it as a new layer.

In [ ]:
# Select markers to visualize
markers_to_plot = ['Ecad', 'CD8', 'CD4']

# Find their indices in the var DataFrame
marker_indices = [np.where(adata.var['marker'] == m)[0][0] for m in markers_to_plot]

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (marker, marker_idx) in enumerate(zip(markers_to_plot, marker_indices)):
    # Get expression values for this marker
    expr_values = adata.layers['exprs'][:, marker_idx]
    
    # Plot histogram
    axes[idx].hist(expr_values, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{marker} Expression (Raw)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Expression Intensity', fontsize=10)
    axes[idx].set_ylabel('Number of Cells', fontsize=10)
    axes[idx].set_yscale('log')  # Log scale for y-axis to see distribution better

plt.tight_layout()
plt.show()

print("Notice: Most cells have low expression, but some have very high values (long tail)")

### Visualizing Raw Expression Distributions

Let's examine the distribution of a few key markers: Ecad (tumor marker), CD8 (cytotoxic T cells), and CD4 (helper T cells).

In [ ]:
# Get marker names from var DataFrame
marker_names = adata.var['marker'].values
print(f"Markers measured: {len(marker_names)}")
print(f"Example markers: {marker_names[:10]}")

## Part 3: Marker Expression Analysis

Protein marker expressions measured by IMC often have highly skewed distributions with long tails. Let's explore this and understand why transformation is necessary.

**Task 1.4:** Create a bar plot showing the total number of cells for each cell type across all images.

Hint: Use `value_counts()` on the celltypes column.

In [ ]:
# Task 1.1: Count unique images
n_images = adata.obs['image'].nunique()
print(f"Number of unique images: {n_images}")

# Task 1.2: Count unique cell types
n_celltypes = adata.obs['celltypes'].nunique()
print(f"Number of cell types: {n_celltypes}")

# Task 1.3: List all cell types
celltypes = adata.obs['celltypes'].unique().tolist()
print(f"\nCell types present:")
for ct in celltypes:
    print(f"  - {ct}")

### Exercise 1: Exploring the Dataset

Complete the following tasks to familiarize yourself with the dataset structure.

**Task 1.1:** How many unique tissue images are in the dataset?

**Task 1.2:** How many unique cell types are present?

**Task 1.3:** What are the names of all cell types?

### Exploring Expression Data (.layers)

Since marker expression values can be represented in different ways (raw counts, normalized, transformed), AnnData stores them in `.layers`.

In [ ]:
# Display marker information
print("Number of markers:", adata.n_vars)
print("\nMarker metadata:")
adata.var.head(10)

### Exploring Feature Metadata (.var)

The `.var` DataFrame contains information about the measured protein markers.

**Key columns in `.obs`:**

- `image` - Identifies which tissue image the cell belongs to
- `sam ple_id` - Sample identifier (patient)
- `Pos_X`, `Pos_Y` - Spatial coordinates of the cell in micrometers
- `celltypes` - Assigned cell type (e.g., Tumor, CD8, CD4)
- `area`, `major_axis_length`, `minor_axis_length` - Morphological features
- Various protein positivity markers (`PD1_pos`, `Ki67_pos`, etc.)

In [ ]:
# Display first few rows of cell metadata
print("Cell metadata columns:")
print(adata.obs.columns.tolist())
print("\nFirst 5 cells:")
adata.obs.head()

### Exploring Cell Metadata (.obs)

The `.obs` DataFrame contains all metadata about individual cells.